# SegmentEveryForam in Google Colab

Test one microscope image at a time. Select **Runtime → Change runtime type → GPU**, then run the cells in order. The U-Net checkpoint comes from Google Drive and the SAM 2.1 checkpoint comes from Meta.

Review the numbered masks in the browser and exclude false detections before exporting. Enter an existing scale calibration or read two pixel coordinates from the scale-bar overlay. This notebook supports reviewing and excluding masks. Use the desktop workflow for drawing, reshaping, or adding masks.

In [ ]:
%pip -q install rtree rasterio gdown plotly
%env SAM2_BUILD_CUDA=0
%pip -q install --no-build-isolation "git+https://github.com/facebookresearch/sam2.git"
%pip -q install --no-deps "git+https://github.com/Geogabriel/SegmentEveryForam.git"

### Load the models

The shared checkpoint link works when Drive allows downloads without sign-in. Otherwise, authorize your Drive and update `drive_checkpoint` if your checkpoint is stored somewhere else.

In [ ]:
from pathlib import Path
from shutil import copyfile
from tempfile import TemporaryDirectory, mkdtemp
from urllib.request import urlretrieve
from zipfile import is_zipfile

import gdown
import keras
import matplotlib.pyplot as plt
import numpy as np
import plotly.express as px
import torch
from google.colab import drive, files
from sam2.build_sam import build_sam2

import segmenteveryforam as sef
from segmenteveryforam.specimen_export import export_specimens

if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime in Colab before loading the models.")

model_dir = Path("/content/sef_models")
model_dir.mkdir(exist_ok=True)
unet_path = model_dir / "best_val_loss.keras"
checkpoint_url = "https://drive.google.com/uc?id=1BnjMogYb1Y_ykKeHBj38yOETX8Yi1rwV"
drive_checkpoint = Path("/content/drive/MyDrive/SEF_v2_100epochs/best_val_loss.keras")

if not is_zipfile(unet_path):
    with TemporaryDirectory(dir=model_dir) as temp_dir:
        pending = Path(temp_dir) / unet_path.name
        try:
            gdown.download(checkpoint_url, str(pending), quiet=False)
        except (gdown.exceptions.FileURLRetrievalError, OSError) as error:
            print(f"Shared checkpoint download failed: {error}")
        if not is_zipfile(pending):
            drive.mount("/content/drive")
            if not drive_checkpoint.is_file():
                raise FileNotFoundError(f"Set drive_checkpoint to your checkpoint location: {drive_checkpoint}")
            copyfile(drive_checkpoint, pending)
        if not is_zipfile(pending):
            raise ValueError("The checkpoint is not a valid .keras archive.")
        pending.replace(unet_path)

unet = keras.saving.load_model(unet_path, compile=False)
sam_path = model_dir / "sam2.1_hiera_large.pt"
if not sam_path.is_file():
    with TemporaryDirectory(dir=model_dir) as temp_dir:
        pending = Path(temp_dir) / sam_path.name
        urlretrieve(
            "https://huggingface.co/facebook/sam2.1-hiera-large/resolve/main/sam2.1_hiera_large.pt",
            pending,
        )
        pending.replace(sam_path)
sam = build_sam2("configs/sam2.1/sam2.1_hiera_l.yaml", str(sam_path), device="cuda")

### Upload and segment one image

In [ ]:
image_dir = Path(mkdtemp(prefix="sef_image_", dir="/content"))
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("Upload exactly one microscope image.")
filename, image_bytes = next(iter(uploaded.items()))
image_path = image_dir / Path(filename).name
image_path.write_bytes(image_bytes)

image = sef.load_image(str(image_path))
polygons, _, boxes = sef.predict_with_box_prompts(image, unet, sam)
detected_grains = sef.polygons_to_grains(polygons, image=image)

fig, ax = plt.subplots(figsize=(14, 10))
sef.plot_image_w_colorful_grains(
    image, [grain.polygon for grain in detected_grains], ax,
    cmap="tab20b", plot_image=True, im_alpha=1.0,
)
for review_id, grain in enumerate(detected_grains, start=1):
    centroid = grain.polygon.centroid
    ax.text(centroid.x, centroid.y, str(review_id), color="yellow", ha="center",
            bbox=dict(facecolor="black", alpha=0.6, pad=1, edgecolor="none"))
ax.set_title(f"Review {len(detected_grains)} detected specimens")
plt.show()

### Calibrate the scale

Run the viewer if you need to measure the scale bar. Zoom in and hover over each end to read its original-image pixel coordinates. Then enter both endpoints and the scale-bar length in micrometers in the export cell. If you already know pixels per meter for this image, enter that value directly and skip the viewer.

In [ ]:
px.imshow(image, binary_string=True, labels=dict(x="Pixel x", y="Pixel y")).show()

### Export measurements, crops, and masks

Enter the sample information and list any unwanted mask numbers in `exclude_review_ids`. The review numbers refer to the overlay above. The CSV and crop filenames use new consecutive `foram_id` values after exclusions.

In [ ]:
sample_id = image_path.stem
species = ""
depth_ccsf = None
age_ma = None
exclude_review_ids = []  # Example: [3, 12]

pixels_per_meter = None  # Enter a known calibration here, or fill in the next three lines.
scale_bar_um = None
scale_start = None  # (x, y) for the first end of the scale bar
scale_end = None    # (x, y) for the second end

if pixels_per_meter is None:
    if any(value is None for value in (scale_bar_um, scale_start, scale_end)):
        raise ValueError("Enter pixels_per_meter or the scale-bar length and both endpoints.")
    if not np.isfinite(scale_bar_um) or scale_bar_um <= 0:
        raise ValueError("Enter a positive scale-bar length in micrometers.")
    pixels_per_meter = np.linalg.norm(np.subtract(scale_end, scale_start)) / (scale_bar_um * 1e-6)
if not np.isfinite(pixels_per_meter) or pixels_per_meter <= 0:
    raise ValueError("Scale calibration must be positive and finite.")

excluded = set(exclude_review_ids)
if excluded.difference(range(1, len(detected_grains) + 1)):
    raise ValueError("An excluded number does not match the review overlay.")
grains = [grain for i, grain in enumerate(detected_grains, start=1) if i not in excluded]
if not grains:
    raise ValueError("There are no specimens to export.")
for grain in grains:
    grain.measure()

metadata = dict(image_path=str(image_path), image_id=image_path.stem, sample_id=sample_id,
                species=species, depth_ccsf=depth_ccsf, Age_Ma=age_ma)
summary = sef.get_summary(grains, pixels_per_meter)
foram_data = sef.get_foram_morphometrics(summary, metadata=metadata)
result_dir = Path(mkdtemp(prefix="sef_results_", dir="/content"))
foram_data_path = result_dir / "foram_data.csv"
foram_data.to_csv(foram_data_path, index=False)
export_dir = export_specimens(
    image, grains, foram_data, metadata, foram_data_path,
    output_dir=result_dir / "specimens",
)
# Use portable references within the downloadable archive.
import pandas as pd
manifest_path = export_dir / "manifest.csv"
manifest = pd.read_csv(manifest_path)
manifest["source_image"] = image_path.name
manifest["morphometrics_csv"] = foram_data_path.relative_to(result_dir).as_posix()
manifest.to_csv(manifest_path, index=False)
print(f"Exported {len(foram_data)} specimens.")
display(foram_data.head())

### Download results

The ZIP contains the measurements, individual crops, binary masks, and manifest. Keep the source microscope image separately.

In [ ]:
from shutil import make_archive

archive_path = make_archive(str(result_dir), "zip", root_dir=result_dir)
files.download(archive_path)